In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from datetime import datetime as dt
import numpy as np

In [ ]:
cpus = pd.read_csv('cpus.csv', index_col=0)

In [ ]:
dates = [dt.strptime(d, '%Y%m%d') for d in cpus.columns]

In [ ]:
colors = [
    'tab:blue',
    'tab:orange',
    'tab:green',
    'tab:red',
    'tab:purple',
    'tab:brown',
    'tab:pink',
    'tab:gray',
    'tab:olive',
    'tab:cyan',
    'rosybrown',
    'peru',
    'fuchsia',
    'yellow',
    'lime'
]

In [ ]:
px = 1/plt.rcParams['figure.dpi']  # pixel in inches
fig = plt.figure(figsize=(1920*px, 1080*px))
ax = fig.add_subplot(111)
for i, (this_serial, row) in enumerate(cpus.iterrows()):
    row_to_plot = row.replace(0, float('nan'))
    color = colors[i]
    ax.scatter(dates, row_to_plot, label=this_serial, color=color)
    median_date = np.median(np.array(dates)[~np.isnan(row_to_plot)].astype('datetime64[D]').astype(float)).astype('datetime64[D]')
    median_sensor = np.median(row_to_plot[~np.isnan(row_to_plot)])
    ax.text(median_date, median_sensor, this_serial, fontsize=12, ha='center', va='center', color='k')
ax.set_xlabel('Date')
ax.set_ylabel('Sensor Number')
ax.legend()
ax.grid()

In [ ]:
all_sensor_nums = np.unique(np.sort(cpus.values.flatten()))

In [ ]:
all_sensor_nums

In [ ]:
sa_metadata = pd.DataFrame(columns=['start_dt','end_dt','sensor_num','cpu_serial'])

In [ ]:
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    cpus_with_sensor = cpus.where(cpus == sensor_num).dropna(how='all').dropna(axis=1, how='all')
    for cpu_id, this_cpu_data in cpus_with_sensor.iterrows():
        this_date_data = (~this_cpu_data.isna())
        date_changes = np.gradient(this_date_data.astype(float))
        end_dates = this_cpu_data.index[(date_changes == -0.5) & (this_date_data)]
        start_dates = this_cpu_data.index[(date_changes == 0.5) & (this_date_data)]
        if this_date_data.iloc[0]:
            start_dates = np.insert(start_dates, 0, this_cpu_data.index[0])
        if this_date_data.iloc[-1]:
            end_dates = np.append(end_dates, this_cpu_data.index[-1])
        if this_date_data.sum() == 1:
            start_dates = np.array([this_cpu_data.index[this_date_data][0]])
            end_dates = np.array([this_cpu_data.index[this_date_data][0]])
        start_dates = [dt.strptime(sd, '%Y%m%d') for sd in start_dates]
        end_dates = [dt.strptime(ed, '%Y%m%d').replace(hour=23, minute=59, second=59) for ed in end_dates]
        print(f'Sensor {sensor_num} was active on CPU {cpu_id} during the following date ranges:')
        for start, end in zip(start_dates, end_dates):
            print(f'  {start} to {end}')
            sa_metadata_for_sensor = {'start_dt' : start, 'end_dt' : end, 'sensor_num' : sensor_num, 'cpu_serial' : cpu_id}
            sa_metadata_for_sensor = pd.DataFrame(sa_metadata_for_sensor, index=[0])
            sa_metadata = pd.concat([sa_metadata, sa_metadata_for_sensor], ignore_index=True)

In [ ]:
for i, conf in sa_metadata.iterrows():
    ax.plot([conf['start_dt'], conf['end_dt']], [conf['sensor_num']+0.1, conf['sensor_num']+0.1], color=colors[np.nonzero(cpus.index == conf['cpu_serial'])[0][0]], alpha=0.5)
    ax.plot([conf['start_dt'], conf['end_dt']], [conf['sensor_num']-0.1, conf['sensor_num']-0.1], color=colors[np.nonzero(cpus.index == conf['cpu_serial'])[0][0]], alpha=0.5)


In [ ]:
sa_metadata = sa_metadata.sort_values(by=['start_dt', 'sensor_num']).reset_index(drop=True)

In [ ]:
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    sa_metadata_for_sensor = sa_metadata[sa_metadata['sensor_num'] == sensor_num]
    start_dts = sa_metadata_for_sensor['start_dt'].values
    end_dts = sa_metadata_for_sensor['end_dt'].values
    if len(start_dts) > 1:
        for i in range(len(start_dts)-1):
            if start_dts[i+1] < end_dts[i]:
                print(f'Warning: Sensor {sensor_num} has overlapping date ranges between\n{start_dts[i].strftime("%Y-%m-%d")} through {end_dts[i].strftime("%Y-%m-%d")} and\n{start_dts[i+1].strftime("%Y-%m-%d")} through {end_dts[i+1].strftime("%Y-%m-%d")}')

In [ ]:
sa_metadata[sa_metadata['sensor_num'] == 9]

In [ ]:
fig

In [ ]:
start_date = dt(2023, 1, 1, 0, 0, 0)
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    sa_metadata_for_sensor = sa_metadata[sa_metadata['sensor_num'] == sensor_num]
    first_entry = np.argmin(sa_metadata_for_sensor['start_dt'])
    sa_metadata.loc[sa_metadata_for_sensor.index[first_entry], 'start_dt'] = start_date

In [ ]:
new_analog_boards = dt(2024, 12, 1, 0, 0, 0)
for i, row in sa_metadata.iterrows():
    if row['start_dt'] < new_analog_boards:
        if row['end_dt'] > new_analog_boards:
            sa_metadata.loc[i, 'end_dt'] = new_analog_boards
            new_row = row.copy()
            new_row['start_dt'] = new_analog_boards
            sa_metadata = pd.concat([sa_metadata, pd.DataFrame(new_row).T], ignore_index=True)

In [ ]:
sa_metadata.sort_values(by=['start_dt', 'sensor_num']).reset_index(drop=True)

In [ ]:
geoscale = np.ones(len(sa_metadata))
geooffset = np.zeros(len(sa_metadata))

staticscale = np.ones(len(sa_metadata))
staticoffset = np.zeros(len(sa_metadata))

massoffset = np.zeros(len(sa_metadata))

In [ ]:
channel_a_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e6, 10e6)
channel_a_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e-9, 10e-9)
channel_a_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 1)
channel_b_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 1e6, 100e6)
channel_b_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e-9, 1e-9)
channel_b_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 10)
channel_c_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e6, 10e6)
channel_c_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 0.1e-6, 0.1e-6)
channel_c_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 1)

In [ ]:
sa_metadata['geo_cal_scale'] = geoscale
sa_metadata['geo_cal_offset'] = geooffset
sa_metadata['static_cal_scale'] = staticscale
sa_metadata['static_cal_offset'] = staticoffset
sa_metadata['mass_cal_offset'] = massoffset
sa_metadata['channel_a_resistor_ohms'] = channel_a_resistor
sa_metadata['channel_a_capacitor_farads'] = channel_a_capacitor
sa_metadata['channel_a_RC_const_seconds'] = channel_a_resistor * channel_a_capacitor
sa_metadata['channel_a_gain'] = channel_a_gain
sa_metadata['channel_b_resistor_ohms'] = channel_b_resistor
sa_metadata['channel_b_capacitor_farads'] = channel_b_capacitor
sa_metadata['channel_b_RC_const_seconds'] = channel_b_resistor * channel_b_capacitor
sa_metadata['channel_b_gain'] = channel_b_gain
sa_metadata['channel_c_resistor_ohms'] = channel_c_resistor
sa_metadata['channel_c_capacitor_farads'] = channel_c_capacitor
sa_metadata['channel_c_RC_const_seconds'] = channel_c_resistor * channel_c_capacitor
sa_metadata['channel_c_gain'] = channel_c_gain

In [ ]:
sa_metadata

In [ ]:
sa_metadata.to_csv('hardware.csv', index=False)